In [2]:
import os
os.chdir("..") # Move To Parent Folder ie Policy Simplifier

from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
from training.config import *
from datasets import load_from_disk
from training.evaluate_metrics import compute_metrics
import torch

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
tokenized_ds=load_from_disk("data/processed_ds")

In [2]:
from transformers import DataCollatorForSeq2Seq

data_collator=DataCollatorForSeq2Seq(model=model,tokenizer=tokenizer)

In [3]:
from transformers import Seq2SeqTrainingArguments,Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(

    output_dir=OUTPUT_DIR,

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,

    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    num_train_epochs=NUM_EPOCHS,

    weight_decay=WEIGHT_DECAY,

    warmup_ratio=WARMUP_RATIO,

    predict_with_generate=False,

    fp16=True,

    logging_steps=50,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,
    
    greater_is_better=False,
    
    metric_for_best_model="eval_loss",
    
    save_total_limit=2,
    
    eval_accumulation_steps=1,
    
    prediction_loss_only=True,

    report_to="none"
)

In [4]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_ds['train'],

    eval_dataset=tokenized_ds['test'],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=None,
)

In [5]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.131700,0.234147


c:\Users\PratushPc\miniconda3\envs\policy_llm\Lib\site-packages\transformers\modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1055, training_loss=0.1739528226061455, metrics={'train_runtime': 9418.1186, 'train_samples_per_second': 0.448, 'train_steps_per_second': 0.112, 'total_flos': 3306578360205312.0, 'train_loss': 0.1739528226061455, 'epoch': 1.0})

In [6]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('models/bart_large_cnn\\tokenizer_config.json',
 'models/bart_large_cnn\\special_tokens_map.json',
 'models/bart_large_cnn\\vocab.json',
 'models/bart_large_cnn\\merges.txt',
 'models/bart_large_cnn\\added_tokens.json',
 'models/bart_large_cnn\\tokenizer.json')

In [8]:
test_model=AutoModelForSeq2SeqLM.from_pretrained("models/bart_large_cnn")
test_tokenizer=AutoTokenizer.from_pretrained("models/bart_large_cnn")

c:\Users\PratushPc\miniconda3\envs\policy_llm\Lib\site-packages\transformers\models\bart\configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


In [4]:
import pandas as pd
PREFIX="summarize privacy policy: "
df=pd.read_csv("data/summaries.csv")

In [ ]:
policy="When you purchase a subscription, we collect your billing address, payment details, and transaction history. Payment card information is processed securely by third-party payment providers and is not stored on our servers. Transaction records may be retained for up to seven years to comply with tax and financial regulations. We use purchase history to recommend subscription plans and improve customer support."

In [6]:
ip=test_tokenizer(PREFIX+policy,max_length=1024,return_tensors="pt",truncation=True)

op=test_model.generate(
    **ip,
    num_beams=4,
    max_new_tokens=128,
    no_repeat_ngram_size=3,
    early_stopping=True
)

c:\Users\PratushPc\miniconda3\envs\policy_llm\Lib\site-packages\transformers\generation\utils.py:1730: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


In [7]:
test_tokenizer.decode(op[0],skip_special_tokens=True)

'When you purchase a subscription, we collect your billing address, payment details, and transaction history.'